# ATHLT Basketball Shot Detector — Training Notebook

**What this does:**
1. Downloads a basketball detection dataset from Roboflow (ball, rim, ball-in-basket classes)
2. Fine-tunes YOLOv11n for 50 epochs
3. Exports to CoreML format (`.mlpackage`) for on-device iOS inference
4. Downloads the model to your computer

**Runtime:** Set to T4 GPU (`Runtime → Change runtime type → T4 GPU`)

**Expected time:** 45–90 minutes on free Colab T4 GPU

## Step 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU available:')
    print(result.stdout[:500])
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')
    print('Training on CPU will take 8+ hours and is not recommended.')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics roboflow --quiet
print('ultralytics and roboflow installed.')

In [ ]:
# Verify ultralytics version — must be >= 8.3 for YOLOv11 support
import ultralytics
print(f'ultralytics version: {ultralytics.__version__}')

from ultralytics import YOLO
import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3 — Download Dataset from Roboflow

**You need a free Roboflow account.** Sign up at roboflow.com (free), then:
1. Go to your account → Settings → Roboflow API → copy your API key
2. Paste it below where it says `YOUR_API_KEY_HERE`

The dataset we use is the basketball detection dataset with these classes:
- `ball` — the basketball
- `ball_in_basket` — ball passing through the rim (makes)
- `basket` — the rim/hoop
- `player` — players
- `player_shooting` — player in shooting motion

If the primary dataset is unavailable, the cell below has fallback datasets to try.

In [ ]:
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # <-- paste your key here

if ROBOFLOW_API_KEY == 'YOUR_API_KEY_HERE':
    raise ValueError('Please set your Roboflow API key above.')

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Primary dataset: Basketball detection with all 5 classes
# Source: used by SwishAI and similar basketball CV projects
# URL: https://universe.roboflow.com/hardik2526-gmail-com/basketball-detection
DATASETS_TO_TRY = [
    ('hardik2526-gmail-com', 'basketball-detection', 1),
    ('basketball-dxhri',     'basketball-1pxce',     4),
    ('basketball-zq14h',     'basketball-lf6dh',     1),
]

dataset = None
for workspace, project_name, version_num in DATASETS_TO_TRY:
    try:
        print(f'Trying {workspace}/{project_name} v{version_num}...')
        project = rf.workspace(workspace).project(project_name)
        dataset = project.version(version_num).download('yolov11')
        print(f'Downloaded: {workspace}/{project_name}')
        break
    except Exception as e:
        print(f'  Failed: {e}')
        continue

if dataset is None:
    print('\nAll primary datasets failed. Trying Roboflow search...')
    # Manual fallback: search for any basketball detection dataset
    try:
        # Search for public datasets
        results = rf.search('basketball detection', type='object-detection')
        for r in results[:3]:
            print(f'  Found: {r}')
        print('\nPick one from the list above and set workspace/project manually below.')
    except:
        print('Search also failed. Check your API key and internet connection.')

print(f'\nDataset location: {dataset.location if dataset else "None"}')

In [ ]:
# Inspect the dataset — check class names and image count
import yaml, os

if dataset is None:
    raise RuntimeError('No dataset downloaded. Fix the step above first.')

data_yaml = os.path.join(dataset.location, 'data.yaml')
with open(data_yaml, 'r') as f:
    data_config = yaml.safe_load(f)

print('Dataset class names:', data_config.get('names', []))
print('Number of classes:', data_config.get('nc', 'unknown'))

# Count images
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f'{split}: {count} images')

print('\ndata.yaml path:', data_yaml)

In [ ]:
# Check if the key classes are present
class_names = data_config.get('names', [])
important_classes = ['ball', 'ball_in_basket', 'basket']

print('Class check:')
for cls in important_classes:
    found = any(cls.lower() in name.lower() for name in class_names)
    status = '✓' if found else '✗ MISSING'
    print(f'  {cls}: {status}')

if not any('ball' in name.lower() for name in class_names):
    print('\nWARNING: No ball class found. This dataset may not be suitable.')
    print('Try a different dataset from DATASETS_TO_TRY above.')

## Step 4 — Fine-tune YOLOv11n

**Model choice:** YOLOv11n (nano) — fastest inference (~17ms/frame on iPhone 13), smallest size (~6MB). Good accuracy for basketball detection.

If you want more accuracy at the cost of speed, change `yolo11n.pt` to `yolo11s.pt`.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11n (downloads from Ultralytics on first run, ~6MB)
model = YOLO('yolo11n.pt')

print('Model loaded:', model.info())

In [ ]:
# Fine-tune on the basketball dataset
# ~50 epochs takes 45-90 minutes on T4 GPU

results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,              # reduce to 8 if you get CUDA OOM errors
    device=0,              # GPU 0 (use 'cpu' if no GPU)
    project='runs/detect',
    name='athlt_shot_detector',
    patience=10,           # stop early if no improvement for 10 epochs
    save=True,
    save_period=10,        # save checkpoint every 10 epochs
    plots=True,
    # Augmentation tuned for basketball court footage
    hsv_s=0.7,             # saturation jitter (helps detect orange ball)
    shear=2.0,             # shear for angle variation
    mixup=0.1,             # helps with crowded scenes
    flipud=0.0,            # don't flip vertically (court orientation matters)
    fliplr=0.5,
)

print('Training complete!')
print(f'Best model saved to: {results.save_dir}/weights/best.pt')

## Step 5 — Validate and review metrics

In [ ]:
# Load the best checkpoint for validation
best_model_path = f'{results.save_dir}/weights/best.pt'
best_model = YOLO(best_model_path)

# Run validation on the held-out validation set
val_results = best_model.val(data=data_yaml, device=0)

print('\n=== Validation Results ===')
print(f'mAP@50:    {val_results.box.map50:.4f}')
print(f'mAP@50-95: {val_results.box.map:.4f}')
print(f'Precision: {val_results.box.mp:.4f}')
print(f'Recall:    {val_results.box.mr:.4f}')

print('\nPer-class AP@50:')
for i, name in enumerate(val_results.names.values()):
    ap = val_results.box.ap50[i] if i < len(val_results.box.ap50) else float('nan')
    print(f'  {name}: {ap:.4f}')

In [ ]:
# Show training plots inline
from IPython.display import Image, display
import os

plot_dir = str(results.save_dir)
plots_to_show = ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']

for plot_name in plots_to_show:
    plot_path = os.path.join(plot_dir, plot_name)
    if os.path.exists(plot_path):
        print(f'--- {plot_name} ---')
        display(Image(filename=plot_path, width=800))
    else:
        print(f'{plot_name}: not found')

## Step 6 — Sample detections on validation images

In [ ]:
import glob
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Pick a few validation images
val_images = glob.glob(os.path.join(dataset.location, 'valid', 'images', '*.jpg'))[:4]
if not val_images:
    val_images = glob.glob(os.path.join(dataset.location, 'valid', 'images', '*.png'))[:4]

if val_images:
    fig, axes = plt.subplots(1, min(4, len(val_images)), figsize=(20, 5))
    if len(val_images) == 1:
        axes = [axes]
    
    CLASS_COLORS = {
        'ball':            'yellow',
        'ball_in_basket':  'lime',
        'basket':          'cyan',
        'player':          'orange',
        'player_shooting': 'magenta',
    }
    DEFAULT_COLOR = 'white'
    
    for ax, img_path in zip(axes, val_images):
        img = PILImage.open(img_path)
        preds = best_model.predict(img_path, conf=0.35, verbose=False)
        
        ax.imshow(np.array(img))
        ax.axis('off')
        
        for pred in preds:
            boxes = pred.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cls_id   = int(box.cls[0])
                cls_name = pred.names[cls_id]
                conf     = float(box.conf[0])
                color    = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)
                
                rect = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 4, f'{cls_name} {conf:.2f}',
                        color=color, fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))
    
    plt.tight_layout()
    plt.savefig('sample_detections.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Sample detections shown above.')
else:
    print('No validation images found for preview.')

## Step 7 — Export to CoreML

This creates `best.mlpackage` — the CoreML model for on-device iOS inference.

`nms=True` includes Non-Maximum Suppression inside the model, reducing post-processing needed on device.

In [ ]:
# Install coremltools if not already available
!pip install coremltools --quiet
print('coremltools ready.')

In [ ]:
import os

print('Exporting to CoreML...')
print('(This takes 2-5 minutes — CoreML export involves model tracing and compilation)')

export_path = best_model.export(
    format='coreml',
    nms=True,        # include NMS in model — simpler inference code on device
    imgsz=640,
    half=False,      # FP32 for better compatibility (FP16 can cause issues on some iPhones)
)

print(f'\nExport complete!')
print(f'CoreML model path: {export_path}')

# Verify the export exists
if os.path.exists(export_path):
    size_mb = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(export_path)
        for f in files
    ) / (1024 * 1024)
    print(f'Model size: {size_mb:.1f} MB')
else:
    print(f'WARNING: Export path does not exist. Check for errors above.')

## Step 8 — Rename and download

The export creates a folder with the original model name. We rename it to `best.mlpackage` for consistency with the ATHLT app's config plugin.

In [ ]:
import shutil

final_path = '/content/best.mlpackage'

# If already at the right path, skip rename
if os.path.abspath(export_path) != os.path.abspath(final_path):
    if os.path.exists(final_path):
        shutil.rmtree(final_path)
    shutil.copytree(export_path, final_path)
    print(f'Copied to {final_path}')
else:
    print(f'Already at {final_path}')

# Verify
if os.path.exists(final_path):
    files = []
    for root, dirs, file_list in os.walk(final_path):
        for f in file_list:
            files.append(os.path.relpath(os.path.join(root, f), final_path))
    print(f'best.mlpackage contains {len(files)} files:')
    for f in files[:10]:
        print(f'  {f}')
    if len(files) > 10:
        print(f'  ... ({len(files) - 10} more)')
else:
    print('ERROR: best.mlpackage not found.')

In [ ]:
# Zip the .mlpackage for download (it's a directory, not a single file)
zip_path = '/content/best_mlpackage.zip'
shutil.make_archive('/content/best_mlpackage', 'zip', '/content', 'best.mlpackage')

zip_size = os.path.getsize(zip_path) / (1024 * 1024)
print(f'Zipped model: {zip_path} ({zip_size:.1f} MB)')

In [ ]:
# Trigger download in the browser
from google.colab import files

print('Downloading best_mlpackage.zip...')
print('A download dialog should appear in your browser.')
print('If it does not, click on the Files tab (left sidebar) and download manually.')
files.download(zip_path)

## Step 9 — What to do next

1. **Unzip** `best_mlpackage.zip` — you'll get a folder called `best.mlpackage`
2. **Move** the folder to `expo/cv/best.mlpackage` in your ATHLT project
3. **Activate the config plugin** in `expo/app.json`:
   ```json
   ["./plugins/withCoreMLModel", { "modelPath": "./cv/best.mlpackage" }]
   ```
4. **Run EAS build**: `eas build --platform ios --profile development`
5. **Test on device**: install the new dev client and open the CV session or Open Run screen

---

**Add `expo/cv/best.mlpackage` to .gitignore** — it's binary and ~20-60MB.

## Optional — Re-train from checkpoint if Colab disconnects

If the session disconnects mid-training, the checkpoints are saved every 10 epochs.
Re-run cells 1-3 (install + dataset), then use this cell instead of the main train cell:

In [ ]:
# Resume from last checkpoint (run this ONLY if training was interrupted)
# Uncomment and run if needed:

# from ultralytics import YOLO
# checkpoint_path = 'runs/detect/athlt_shot_detector/weights/last.pt'
# model = YOLO(checkpoint_path)
# results = model.train(resume=True)